# 03 — Correct verifier data provenance and grouped splits

This replaces the legacy example-level 98%/2% split. Every harmony record must expose a patient or source-record grouping key. Evaluation patients, source IDs, and exact report hashes are removed, then groups are assigned 80%/10%/10% to train/validation/test.

In [ ]:
from pathlib import Path
import json, os, sys

def find_rerun_dir():
    candidates = [
        Path(os.environ.get("JAMIA_RERUN_DIR", "")),
        Path.cwd(),
        Path.cwd().parent,
    ]
    for candidate in candidates:
        if str(candidate) and (candidate / "rerun_config.json").exists():
            return candidate.resolve()
    raise FileNotFoundError("Set JAMIA_RERUN_DIR to the folder containing rerun_config.json")

RERUN_DIR = find_rerun_dir()
# Never expose the implementation directory as a top-level import location:
# rerun_code/statistics.py would shadow Python's standard-library statistics.
implementation_dir = (RERUN_DIR / "src" / "rerun_code").resolve()
clean_sys_path = []
for entry in sys.path:
    try:
        resolved_entry = Path(entry or ".").resolve()
    except Exception:
        resolved_entry = None
    if resolved_entry != implementation_dir:
        clean_sys_path.append(entry)
sys.path[:] = clean_sys_path
sys.path.insert(0, str(RERUN_DIR / "src"))
from rerun_code.config import load_config, output_paths
RERUN_DIR, CONFIG = load_config(RERUN_DIR)
PATHS = output_paths(CONFIG)
print("Code:", RERUN_DIR)
print("Output:", PATHS["root"])

In [ ]:
import pandas as pd
from rerun_code.common import read_jsonl, write_jsonl, write_json
from rerun_code.verifier_data import standardize_verifier_records, remove_evaluation_overlap, grouped_stratified_split, assert_group_disjoint, split_summary

source = Path(CONFIG["verifier_data"]["source_jsonl"])
if not source.exists():
    raise FileNotFoundError(f"Set verifier_data.source_jsonl to the harmony/verifier JSONL with patient/source provenance: {source}")
source_records = read_jsonl(source)
standardized = standardize_verifier_records(source_records)
print("Parsed verifier targets:", standardized["verdict_source"].value_counts().to_dict())
print("Grouping provenance:", standardized["grouping_level"].value_counts().to_dict())
print("Tasks:", standardized["task"].value_counts().to_dict())
evaluation = pd.concat([pd.DataFrame(read_jsonl(PATHS["manifests"] / d / "fixed_queries.jsonl")) for d in ("mimic", "iuhn")], ignore_index=True)
clean, excluded = remove_evaluation_overlap(standardized, evaluation)
split = grouped_stratified_split(clean, CONFIG["verifier_data"]["split_fractions"], seed=CONFIG["verifier_data"]["seed"])
assert_group_disjoint(split)
summary = split_summary(split)
if summary.get("test", {}).get("n_examples", 0) < CONFIG["verifier_data"]["minimum_test_examples"]:
    raise AssertionError(f"Verifier test split is too small: {summary}")
write_jsonl(PATHS["verifier_data"] / "all_grouped.jsonl", split.to_dict("records"))
write_jsonl(PATHS["verifier_data"] / "excluded_evaluation_overlap.jsonl", excluded.to_dict("records"))
for name in ("train", "validation", "test"):
    write_jsonl(PATHS["verifier_data"] / f"{name}.jsonl", split.loc[split["split"] == name].to_dict("records"))
write_json(PATHS["verifier_data"] / "split_audit.json", {
    "source": str(source), "n_source": len(standardized), "n_excluded": len(excluded),
    "verdict_sources": standardized["verdict_source"].value_counts().to_dict(),
    "grouping_levels": standardized["grouping_level"].value_counts().to_dict(),
    "tasks": standardized["task"].value_counts().to_dict(), "summary": summary,
    "limitation": "The supplied harmony file has source image/report IDs but no patient IDs; splitting is source-disjoint rather than demonstrably patient-disjoint."
})
print(json.dumps(summary, indent=2)); print("VERIFIER SPLIT GATE PASSED")